In [ ]:
# =========================================================
# LEAKAGE-FREE CATBOOST TRAINING
# =========================================================

# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported")

# =========================================================
# 4. LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

# =========================================================
# 5. REMOVE UNWANTED COLUMNS
# =========================================================

drop_cols = [

    'Index',

    'geohash_demand_mean',
    'hour_demand_mean',
    'roadtype_demand_mean',
    'day_demand_mean'
]

train_df.drop(columns=drop_cols, inplace=True)

test_df.drop(columns=drop_cols, inplace=True, errors='ignore')

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Leaky Features Removed")

# =========================================================
# 6. PREPARE FEATURES
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

groups = train_df['geohash']

print("Feature Shape :", X.shape)

# =========================================================
# 7. CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

# =========================================================
# 8. CONVERT CATEGORICAL FEATURES
# =========================================================

for col in categorical_features:

    X[col] = X[col].astype(str)

    test_df[col] = test_df[col].astype(str)

print("Categorical Conversion Done")

# =========================================================
# 9. GROUP KFOLD
# =========================================================

gkf = GroupKFold(n_splits=5)

scores = []

test_preds = np.zeros(len(test_df))

# =========================================================
# 10. TRAINING LOOP
# =========================================================

for fold, (train_idx, valid_idx) in enumerate(

    gkf.split(X, y, groups)
):

    print("\n")
    print("="*50)
    print(f"FOLD {fold+1}")
    print("="*50)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # =====================================================
    # MODEL
    # =====================================================

    model = CatBoostRegressor(

        iterations=5000,

        learning_rate=0.03,

        depth=8,

        loss_function='RMSE',

        eval_metric='RMSE',

        task_type='GPU',

        devices='0',

        random_seed=42,

        verbose=500
    )

    # =====================================================
    # TRAIN
    # =====================================================

    model.fit(

        X_train,
        y_train,

        cat_features=categorical_features,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        use_best_model=True
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    valid_preds = model.predict(X_valid)

    r2 = r2_score(y_valid, valid_preds)

    scores.append(r2)

    print(f"\nFold R² : {r2}")

    # =====================================================
    # TEST PREDICTIONS
    # =====================================================

    fold_test_preds = model.predict(test_df)

    test_preds += fold_test_preds / 5

# =========================================================
# 11. FINAL SCORE
# =========================================================

print("\n")
print("="*60)
print("CROSS VALIDATION RESULTS")
print("="*60)

print("Fold Scores :", scores)

print("Mean R² :", np.mean(scores))

print("Competition Score :", 100 * np.mean(scores))

# =========================================================
# 12. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': test_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 13. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_leakage_free.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\nSubmission Saved Successfully")

print(submission_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Leaky Features Removed
Feature Shape : (77299, 31)
Categorical Conversion Done


FOLD 1
0:	learn: 0.1448520	test: 0.1108578	best: 0.1108578 (0)	total: 34.8ms	remaining: 2m 53s
bestTest = 0.05520284756
bestIteration = 89
Shrink model to first 90 iterations.

Fold R² : 0.7611563402598583


FOLD 2
0:	learn: 0.1383686	test: 0.1405443	best: 0.1405443 (0)	total: 67.9ms	remaining: 5m 39s
500:	learn: 0.0335486	test: 0.0746002	best: 0.0733634 (240)	total: 14.2s	remaining: 2m 7s
bestTest = 0.07336343356
bestIteration = 240
Shrink model to first 241 iterations.

Fold R² : 0.7396648469209764


FOLD 3
0:	learn: 0.1379450	test: 0.1419517	best: 0.1419517 (0)	total: 22.1ms	remaining: 1m 50s
500:	learn: 0.0335502	test: 0.0787706	best: 0.0778870 (210)	total: 14.4s	remaining: 2m 8s
bestTest = 0.07788696822
b

In [ ]:
# =========================================================
# OOF TARGET ENCODING + CATBOOST
# =========================================================

# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# =========================================================
# 4. LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

# =========================================================
# 5. REMOVE OLD LEAKY FEATURES
# =========================================================

leaky_cols = [

    'geohash_demand_mean',
    'hour_demand_mean',
    'roadtype_demand_mean',
    'day_demand_mean'
]

train_df.drop(columns=leaky_cols, inplace=True)

test_df.drop(columns=leaky_cols, inplace=True, errors='ignore')

print("Old Leaky Features Removed")

# =========================================================
# 6. REMOVE INDEX
# =========================================================

if 'Index' in train_df.columns:
    train_df.drop(columns=['Index'], inplace=True)

if 'Index' in test_df.columns:
    test_df.drop(columns=['Index'], inplace=True)

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Index Removed")

# =========================================================
# 7. OOF TARGET ENCODING FUNCTION
# =========================================================

def create_oof_target_encoding(

    train,
    test,
    column,
    target,
    n_splits=5
):

    train_encoded = np.zeros(len(train))

    test_encoded = np.zeros(len(test))

    global_mean = train[target].mean()

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(train):

        X_train = train.iloc[train_idx]

        X_valid = train.iloc[valid_idx]

        means = X_train.groupby(column)[target].mean()

        train_encoded[valid_idx] = (
            X_valid[column]
            .map(means)
            .fillna(global_mean)
        )

    full_means = train.groupby(column)[target].mean()

    test_encoded = (
        test[column]
        .map(full_means)
        .fillna(global_mean)
    )

    return train_encoded, test_encoded

# =========================================================
# 8. CREATE OOF TARGET ENCODINGS
# =========================================================

target_cols = [

    'geohash',
    'hour',
    'RoadType',
    'day'
]

target_feature_names = [

    'geohash_te',
    'hour_te',
    'roadtype_te',
    'day_te'
]

# =========================================================
# 9. GENERATE OOF FEATURES
# =========================================================

for col, new_col in zip(target_cols, target_feature_names):

    print(f"\nCreating {new_col}")

    train_df[new_col], test_df[new_col] = (
        create_oof_target_encoding(

            train=train_df,

            test=test_df,

            column=col,

            target='demand',

            n_splits=5
        )
    )

print("\nOOF Target Encoding Completed")

# =========================================================
# 10. PREPARE FEATURES
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

print("Feature Shape :", X.shape)

# =========================================================
# 11. CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

# =========================================================
# 12. CONVERT CATEGORICAL FEATURES
# =========================================================

for col in categorical_features:

    X[col] = X[col].astype(str)

    test_df[col] = test_df[col].astype(str)

print("Categorical Conversion Completed")

# =========================================================
# 13. KFOLD CROSS VALIDATION
# =========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

test_preds = np.zeros(len(test_df))

# =========================================================
# 14. TRAINING LOOP
# =========================================================

for fold, (train_idx, valid_idx) in enumerate(

    kf.split(X)
):

    print("\n")
    print("="*50)
    print(f"FOLD {fold+1}")
    print("="*50)

    X_train = X.iloc[train_idx]

    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    y_valid = y.iloc[valid_idx]

    # =====================================================
    # MODEL
    # =====================================================

    model = CatBoostRegressor(

        iterations=5000,

        learning_rate=0.03,

        depth=8,

        loss_function='RMSE',

        eval_metric='RMSE',

        task_type='GPU',

        devices='0',

        random_seed=42,

        verbose=500
    )

    # =====================================================
    # TRAIN
    # =====================================================

    model.fit(

        X_train,
        y_train,

        cat_features=categorical_features,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        use_best_model=True
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    valid_preds = model.predict(X_valid)

    r2 = r2_score(y_valid, valid_preds)

    scores.append(r2)

    print(f"\nFold R² : {r2}")

    # =====================================================
    # TEST PREDICTIONS
    # =====================================================

    fold_test_preds = model.predict(test_df)

    test_preds += fold_test_preds / 5

# =========================================================
# 15. FINAL RESULTS
# =========================================================

print("\n")
print("="*60)
print("FINAL CROSS VALIDATION RESULTS")
print("="*60)

print("Fold Scores :", scores)

print("Mean R² :", np.mean(scores))

print("Competition Score :", 100 * np.mean(scores))

# =========================================================
# 16. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': test_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 17. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_oof_target_encoding.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\nSubmission Saved Successfully")

print(submission_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported Successfully
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Old Leaky Features Removed
Index Removed

Creating geohash_te

Creating hour_te

Creating roadtype_te

Creating day_te

OOF Target Encoding Completed
Feature Shape : (77299, 35)
Categorical Conversion Completed


FOLD 1
0:	learn: 0.1384535	test: 0.1385291	best: 0.1385291 (0)	total: 95.2ms	remaining: 7m 56s
500:	learn: 0.0324073	test: 0.0348179	best: 0.0348179 (500)	total: 14.7s	remaining: 2m 11s
1000:	learn: 0.0292343	test: 0.0334556	best: 0.0334556 (1000)	total: 29s	remaining: 1m 55s
1500:	learn: 0.0273502	test: 0.0327743	best: 0.0327718 (1491)	total: 44.1s	remaining: 1m 42s
2000:	learn: 0.0259920	test: 0.0323710	best: 0.0323710 (2000)	total: 59.5s	remaining: 1m 29s
2500:	learn: 0.0248670	test: 0.0320973	best: 0.0320964 (2498)	total: 1m 15s	remaining: 1m 15s
3000:	learn: 0.02

In [ ]:
# =========================================================
# FINAL ENSEMBLE MODEL
# CATBOOST + XGBOOST + LIGHTGBM
# UPDATED FOR NEW GPU LIBRARIES
# =========================================================

# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost xgboost lightgbm -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# =========================================================
# 4. LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

# =========================================================
# 5. REMOVE OLD LEAKY FEATURES
# =========================================================

leaky_cols = [

    'geohash_demand_mean',
    'hour_demand_mean',
    'roadtype_demand_mean',
    'day_demand_mean'
]

train_df.drop(columns=leaky_cols, inplace=True)

test_df.drop(columns=leaky_cols, inplace=True, errors='ignore')

print("Old Leaky Features Removed")

# =========================================================
# 6. REMOVE INDEX
# =========================================================

if 'Index' in train_df.columns:
    train_df.drop(columns=['Index'], inplace=True)

if 'Index' in test_df.columns:
    test_df.drop(columns=['Index'], inplace=True)

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Index Removed")

# =========================================================
# 7. OOF TARGET ENCODING FUNCTION
# =========================================================

def create_oof_target_encoding(

    train,
    test,
    column,
    target,
    n_splits=5
):

    train_encoded = np.zeros(len(train))

    test_encoded = np.zeros(len(test))

    global_mean = train[target].mean()

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(train):

        X_train_fold = train.iloc[train_idx]

        X_valid_fold = train.iloc[valid_idx]

        means = X_train_fold.groupby(column)[target].mean()

        train_encoded[valid_idx] = (
            X_valid_fold[column]
            .map(means)
            .fillna(global_mean)
        )

    full_means = train.groupby(column)[target].mean()

    test_encoded = (
        test[column]
        .map(full_means)
        .fillna(global_mean)
    )

    return train_encoded, test_encoded

# =========================================================
# 8. CREATE OOF TARGET ENCODINGS
# =========================================================

target_cols = [

    'geohash',
    'hour',
    'RoadType',
    'day'
]

target_feature_names = [

    'geohash_te',
    'hour_te',
    'roadtype_te',
    'day_te'
]

for col, new_col in zip(target_cols, target_feature_names):

    print(f"Creating {new_col}")

    train_df[new_col], test_df[new_col] = (
        create_oof_target_encoding(

            train=train_df,

            test=test_df,

            column=col,

            target='demand',

            n_splits=5
        )
    )

print("OOF Target Encoding Completed")

# =========================================================
# 9. PREPARE FEATURES
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

print("Feature Shape :", X.shape)

# =========================================================
# 10. CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

# =========================================================
# 11. CONVERT TO STRING
# =========================================================

for col in categorical_features:

    X[col] = X[col].astype(str)

    test_df[col] = test_df[col].astype(str)

print("String Conversion Completed")

# =========================================================
# 12. LABEL ENCODING
# =========================================================

for col in categorical_features:

    le = LabelEncoder()

    combined_vals = (
        list(X[col]) +
        list(test_df[col])
    )

    le.fit(combined_vals)

    X[col] = le.transform(X[col])

    test_df[col] = le.transform(test_df[col])

print("Label Encoding Completed")

# =========================================================
# 13. KFOLD SETUP
# =========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================================================
# 14. PREDICTION ARRAYS
# =========================================================

cat_preds = np.zeros(len(test_df))

xgb_preds = np.zeros(len(test_df))

lgb_preds = np.zeros(len(test_df))

cat_scores = []

xgb_scores = []

lgb_scores = []

# =========================================================
# 15. TRAINING LOOP
# =========================================================

for fold, (train_idx, valid_idx) in enumerate(

    kf.split(X)
):

    print("\n")
    print("="*60)
    print(f"FOLD {fold+1}")
    print("="*60)

    X_train = X.iloc[train_idx]

    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    y_valid = y.iloc[valid_idx]

    # =====================================================
    # CATBOOST
    # =====================================================

    print("\nTraining CatBoost...")

    cat_model = CatBoostRegressor(

        iterations=3000,

        learning_rate=0.03,

        depth=8,

        loss_function='RMSE',

        eval_metric='RMSE',

        task_type='GPU',

        devices='0',

        random_seed=42,

        verbose=0
    )

    cat_model.fit(

        X_train,
        y_train,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        verbose=False
    )

    cat_valid_preds = cat_model.predict(X_valid)

    cat_r2 = r2_score(
        y_valid,
        cat_valid_preds
    )

    cat_scores.append(cat_r2)

    cat_preds += (
        cat_model.predict(test_df) / 5
    )

    print(f"CatBoost R² : {cat_r2}")

    # =====================================================
    # XGBOOST
    # =====================================================

    print("\nTraining XGBoost...")

    xgb_model = XGBRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        objective='reg:squarederror',

        tree_method='hist',

        device='cuda',

        random_state=42
    )

    xgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)],

        verbose=False
    )

    xgb_valid_preds = xgb_model.predict(X_valid)

    xgb_r2 = r2_score(
        y_valid,
        xgb_valid_preds
    )

    xgb_scores.append(xgb_r2)

    xgb_preds += (
        xgb_model.predict(test_df) / 5
    )

    print(f"XGBoost R² : {xgb_r2}")

    # =====================================================
    # LIGHTGBM
    # =====================================================

    print("\nTraining LightGBM...")

    lgb_model = LGBMRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        device='gpu'
    )

    lgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)]
    )

    lgb_valid_preds = lgb_model.predict(X_valid)

    lgb_r2 = r2_score(
        y_valid,
        lgb_valid_preds
    )

    lgb_scores.append(lgb_r2)

    lgb_preds += (
        lgb_model.predict(test_df) / 5
    )

    print(f"LightGBM R² : {lgb_r2}")

# =========================================================
# 16. FINAL MODEL SCORES
# =========================================================

print("\n")
print("="*60)
print("FINAL MODEL SCORES")
print("="*60)

print(f"CatBoost Mean R² : {np.mean(cat_scores)}")

print(f"XGBoost Mean R² : {np.mean(xgb_scores)}")

print(f"LightGBM Mean R² : {np.mean(lgb_scores)}")

# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.20 * cat_preds +

    0.50 * xgb_preds +

    0.30 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_ensemble.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported Successfully
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Old Leaky Features Removed
Index Removed
Creating geohash_te
Creating hour_te
Creating roadtype_te
Creating day_te
OOF Target Encoding Completed
Feature Shape : (77299, 35)
String Conversion Completed
Label Encoding Completed


FOLD 1

Training CatBoost...
CatBoost R² : 0.9408342673708273

Training XGBoost...
XGBoost R² : 0.9509786864026983

Training LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1349
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 35
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 27 de

In [ ]:
# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.20 * cat_preds +

    0.50 * xgb_preds +

    0.30 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_ensemble.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)


Ensemble Predictions Created

Submission Sample
   Index    demand
0      0  0.054879
1      1  0.028542
2      2  0.026159
3      3  0.021149
4      4  0.045390


SUBMISSION SAVED SUCCESSFULLY
/content/drive/MyDrive/Traffic_Prediction/submission_ensemble.csv


In [ ]:
# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.10 * cat_preds +

    0.65 * xgb_preds +

    0.25 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionA.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)


Ensemble Predictions Created

Submission Sample
   Index    demand
0      0  0.057280
1      1  0.029395
2      2  0.029469
3      3  0.020348
4      4  0.045630


SUBMISSION SAVED SUCCESSFULLY
/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionA.csv


In [ ]:
# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.05 * cat_preds +

    0.70 * xgb_preds +

    0.25 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionC.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)


Ensemble Predictions Created

Submission Sample
   Index    demand
0      0  0.058144
1      1  0.029693
2      2  0.030485
3      3  0.020156
4      4  0.045841


SUBMISSION SAVED SUCCESSFULLY
/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionC.csv


In [ ]:
# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.15 * cat_preds +

    0.60 * xgb_preds +

    0.25 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionB.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)


Ensemble Predictions Created

Submission Sample
   Index    demand
0      0  0.056415
1      1  0.029098
2      2  0.028453
3      3  0.020540
4      4  0.045418


SUBMISSION SAVED SUCCESSFULLY
/content/drive/MyDrive/Traffic_Prediction/submission_ensemble_versionB.csv


In [ ]:
# =========================================================
# STACKING ENSEMBLE
# CATBOOST + XGBOOST + LIGHTGBM + RIDGE
# =========================================================

# =========================================================
# 1. MOUNT DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost xgboost lightgbm -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# =========================================================
# 4. LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

# =========================================================
# 5. REMOVE LEAKY FEATURES
# =========================================================

leaky_cols = [

    'geohash_demand_mean',
    'hour_demand_mean',
    'roadtype_demand_mean',
    'day_demand_mean'
]

train_df.drop(columns=leaky_cols, inplace=True)

test_df.drop(columns=leaky_cols, inplace=True, errors='ignore')

# =========================================================
# 6. REMOVE INDEX
# =========================================================

if 'Index' in train_df.columns:
    train_df.drop(columns=['Index'], inplace=True)

if 'Index' in test_df.columns:
    test_df.drop(columns=['Index'], inplace=True)

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Cleanup Completed")

# =========================================================
# 7. OOF TARGET ENCODING
# =========================================================

def create_oof_target_encoding(

    train,
    test,
    column,
    target,
    n_splits=5
):

    train_encoded = np.zeros(len(train))

    test_encoded = np.zeros(len(test))

    global_mean = train[target].mean()

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(train):

        X_train_fold = train.iloc[train_idx]

        X_valid_fold = train.iloc[valid_idx]

        means = X_train_fold.groupby(column)[target].mean()

        train_encoded[valid_idx] = (
            X_valid_fold[column]
            .map(means)
            .fillna(global_mean)
        )

    full_means = train.groupby(column)[target].mean()

    test_encoded = (
        test[column]
        .map(full_means)
        .fillna(global_mean)
    )

    return train_encoded, test_encoded

# =========================================================
# 8. CREATE OOF TARGET ENCODINGS
# =========================================================

target_cols = [

    'geohash',
    'hour',
    'RoadType',
    'day'
]

target_feature_names = [

    'geohash_te',
    'hour_te',
    'roadtype_te',
    'day_te'
]

for col, new_col in zip(target_cols, target_feature_names):

    print(f"Creating {new_col}")

    train_df[new_col], test_df[new_col] = (
        create_oof_target_encoding(

            train=train_df,

            test=test_df,

            column=col,

            target='demand',

            n_splits=5
        )
    )

print("OOF Encoding Completed")

# =========================================================
# 9. PREPARE FEATURES
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

# =========================================================
# 10. CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

# =========================================================
# 11. CONVERT TO STRING
# =========================================================

for col in categorical_features:

    X[col] = X[col].astype(str)

    test_df[col] = test_df[col].astype(str)

# =========================================================
# 12. LABEL ENCODING
# =========================================================

for col in categorical_features:

    le = LabelEncoder()

    combined_vals = (
        list(X[col]) +
        list(test_df[col])
    )

    le.fit(combined_vals)

    X[col] = le.transform(X[col])

    test_df[col] = le.transform(test_df[col])

print("Label Encoding Completed")

# =========================================================
# 13. KFOLD
# =========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================================================
# 14. OOF ARRAYS
# =========================================================

oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))

test_cat = np.zeros(len(test_df))
test_xgb = np.zeros(len(test_df))
test_lgb = np.zeros(len(test_df))

# =========================================================
# 15. TRAINING LOOP
# =========================================================

for fold, (train_idx, valid_idx) in enumerate(

    kf.split(X)
):

    print("\n")
    print("="*60)
    print(f"FOLD {fold+1}")
    print("="*60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    # =====================================================
    # CATBOOST
    # =====================================================

    print("\nTraining CatBoost...")

    cat_model = CatBoostRegressor(

        iterations=3000,

        learning_rate=0.03,

        depth=8,

        loss_function='RMSE',

        eval_metric='RMSE',

        task_type='GPU',

        devices='0',

        random_seed=42,

        verbose=0
    )

    cat_model.fit(

        X_train,
        y_train,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        verbose=False
    )

    cat_valid = cat_model.predict(X_valid)

    oof_cat[valid_idx] = cat_valid

    test_cat += (
        cat_model.predict(test_df) / 5
    )

    # =====================================================
    # XGBOOST
    # =====================================================

    print("Training XGBoost...")

    xgb_model = XGBRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        objective='reg:squarederror',

        tree_method='hist',

        device='cuda',

        random_state=42
    )

    xgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)],

        verbose=False
    )

    xgb_valid = xgb_model.predict(X_valid)

    oof_xgb[valid_idx] = xgb_valid

    test_xgb += (
        xgb_model.predict(test_df) / 5
    )

    # =====================================================
    # LIGHTGBM
    # =====================================================

    print("Training LightGBM...")

    lgb_model = LGBMRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        device='gpu'
    )

    lgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)]
    )

    lgb_valid = lgb_model.predict(X_valid)

    oof_lgb[valid_idx] = lgb_valid

    test_lgb += (
        lgb_model.predict(test_df) / 5
    )

# =========================================================
# 16. CREATE STACKING DATASET
# =========================================================

stack_train = pd.DataFrame({

    'catboost': oof_cat,

    'xgboost': oof_xgb,

    'lightgbm': oof_lgb
})

stack_test = pd.DataFrame({

    'catboost': test_cat,

    'xgboost': test_xgb,

    'lightgbm': test_lgb
})

print("\nStacking Dataset Created")

# =========================================================
# 17. META MODEL
# =========================================================

meta_model = Ridge(alpha=1.0)

meta_model.fit(

    stack_train,
    y
)

# =========================================================
# 18. STACKING PREDICTIONS
# =========================================================

stack_preds = meta_model.predict(stack_test)

print("Stacking Predictions Created")

# =========================================================
# 19. VALIDATION SCORE
# =========================================================

stack_oof = meta_model.predict(stack_train)

stack_r2 = r2_score(y, stack_oof)

print("\n")
print("="*60)
print("STACKING RESULTS")
print("="*60)

print(f"Stacking R² : {stack_r2}")

# =========================================================
# 20. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': stack_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 21. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_stacking.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("STACKING SUBMISSION SAVED")
print("="*60)

print(submission_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported Successfully
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Cleanup Completed
Creating geohash_te
Creating hour_te
Creating roadtype_te
Creating day_te
OOF Encoding Completed
Label Encoding Completed


FOLD 1

Training CatBoost...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1349
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 35
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 27 dense feature groups (1.65 MB) transferred to GPU in 0.003172 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.093784
[LightGBM]

In [ ]:
# =========================================================
# FINAL ENSEMBLE WITH XGB-A TUNING
# =========================================================

# =========================================================
# 1. MOUNT DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost xgboost lightgbm -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# =========================================================
# 4. LOAD DATA
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)

test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)

print("Test Shape  :", test_df.shape)

# =========================================================
# 5. REMOVE OLD LEAKY FEATURES
# =========================================================

leaky_cols = [

    'geohash_demand_mean',
    'hour_demand_mean',
    'roadtype_demand_mean',
    'day_demand_mean'
]

train_df.drop(columns=leaky_cols, inplace=True)

test_df.drop(columns=leaky_cols, inplace=True, errors='ignore')

print("Old Leaky Features Removed")

# =========================================================
# 6. REMOVE INDEX
# =========================================================

if 'Index' in train_df.columns:
    train_df.drop(columns=['Index'], inplace=True)

if 'Index' in test_df.columns:
    test_df.drop(columns=['Index'], inplace=True)

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Cleanup Completed")

# =========================================================
# 7. OOF TARGET ENCODING FUNCTION
# =========================================================

def create_oof_target_encoding(

    train,
    test,
    column,
    target,
    n_splits=5
):

    train_encoded = np.zeros(len(train))

    test_encoded = np.zeros(len(test))

    global_mean = train[target].mean()

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(train):

        X_train_fold = train.iloc[train_idx]

        X_valid_fold = train.iloc[valid_idx]

        means = X_train_fold.groupby(column)[target].mean()

        train_encoded[valid_idx] = (

            X_valid_fold[column]
            .map(means)
            .fillna(global_mean)
        )

    full_means = train.groupby(column)[target].mean()

    test_encoded = (

        test[column]
        .map(full_means)
        .fillna(global_mean)
    )

    return train_encoded, test_encoded

# =========================================================
# 8. CREATE OOF TARGET ENCODINGS
# =========================================================

target_cols = [

    'geohash',
    'hour',
    'RoadType',
    'day'
]

target_feature_names = [

    'geohash_te',
    'hour_te',
    'roadtype_te',
    'day_te'
]

for col, new_col in zip(target_cols, target_feature_names):

    print(f"Creating {new_col}")

    train_df[new_col], test_df[new_col] = (

        create_oof_target_encoding(

            train=train_df,

            test=test_df,

            column=col,

            target='demand',

            n_splits=5
        )
    )

print("OOF Target Encoding Completed")

# =========================================================
# 9. PREPARE FEATURES
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

print("Feature Shape :", X.shape)

# =========================================================
# 10. CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

# =========================================================
# 11. CONVERT TO STRING
# =========================================================

for col in categorical_features:

    X[col] = X[col].astype(str)

    test_df[col] = test_df[col].astype(str)

print("String Conversion Completed")

# =========================================================
# 12. LABEL ENCODING
# =========================================================

for col in categorical_features:

    le = LabelEncoder()

    combined_vals = (

        list(X[col]) +

        list(test_df[col])
    )

    le.fit(combined_vals)

    X[col] = le.transform(X[col])

    test_df[col] = le.transform(test_df[col])

print("Label Encoding Completed")

# =========================================================
# 13. KFOLD SETUP
# =========================================================

kf = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

# =========================================================
# 14. PREDICTION ARRAYS
# =========================================================

cat_preds = np.zeros(len(test_df))

xgb_preds = np.zeros(len(test_df))

lgb_preds = np.zeros(len(test_df))

cat_scores = []

xgb_scores = []

lgb_scores = []

# =========================================================
# 15. TRAINING LOOP
# =========================================================

for fold, (train_idx, valid_idx) in enumerate(

    kf.split(X)
):

    print("\n")
    print("="*60)
    print(f"FOLD {fold+1}")
    print("="*60)

    X_train = X.iloc[train_idx]

    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    y_valid = y.iloc[valid_idx]

    # =====================================================
    # CATBOOST
    # =====================================================

    print("\nTraining CatBoost...")

    cat_model = CatBoostRegressor(

        iterations=3000,

        learning_rate=0.03,

        depth=8,

        loss_function='RMSE',

        eval_metric='RMSE',

        task_type='GPU',

        devices='0',

        random_seed=42,

        verbose=0
    )

    cat_model.fit(

        X_train,
        y_train,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        verbose=False
    )

    cat_valid_preds = cat_model.predict(X_valid)

    cat_r2 = r2_score(
        y_valid,
        cat_valid_preds
    )

    cat_scores.append(cat_r2)

    cat_preds += (
        cat_model.predict(test_df) / 5
    )

    print(f"CatBoost R² : {cat_r2}")

    # =====================================================
    # XGBOOST - VERSION XGB-A
    # =====================================================

    print("\nTraining Tuned XGBoost...")

    xgb_model = XGBRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=9,

        subsample=0.9,

        colsample_bytree=0.9,

        min_child_weight=3,

        reg_lambda=2,

        objective='reg:squarederror',

        tree_method='hist',

        device='cuda',

        random_state=42
    )

    xgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)],

        verbose=False
    )

    xgb_valid_preds = xgb_model.predict(X_valid)

    xgb_r2 = r2_score(
        y_valid,
        xgb_valid_preds
    )

    xgb_scores.append(xgb_r2)

    xgb_preds += (
        xgb_model.predict(test_df) / 5
    )

    print(f"XGBoost R² : {xgb_r2}")

    # =====================================================
    # LIGHTGBM
    # =====================================================

    print("\nTraining LightGBM...")

    lgb_model = LGBMRegressor(

        n_estimators=3000,

        learning_rate=0.03,

        max_depth=8,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        device='gpu'
    )

    lgb_model.fit(

        X_train,
        y_train,

        eval_set=[(X_valid, y_valid)]
    )

    lgb_valid_preds = lgb_model.predict(X_valid)

    lgb_r2 = r2_score(
        y_valid,
        lgb_valid_preds
    )

    lgb_scores.append(lgb_r2)

    lgb_preds += (
        lgb_model.predict(test_df) / 5
    )

    print(f"LightGBM R² : {lgb_r2}")

# =========================================================
# 16. FINAL MODEL SCORES
# =========================================================

print("\n")
print("="*60)
print("FINAL MODEL SCORES")
print("="*60)

print(f"CatBoost Mean R² : {np.mean(cat_scores)}")

print(f"XGBoost Mean R² : {np.mean(xgb_scores)}")

print(f"LightGBM Mean R² : {np.mean(lgb_scores)}")

# =========================================================
# 17. FINAL ENSEMBLE
# =========================================================

final_preds = (

    0.05 * cat_preds +

    0.70 * xgb_preds +

    0.25 * lgb_preds
)

print("\nEnsemble Predictions Created")

# =========================================================
# 18. CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_preds
})

print("\nSubmission Sample")

print(submission.head())

# =========================================================
# 19. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_xgbA_ensemble.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*60)

print(submission_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported Successfully
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Old Leaky Features Removed
Cleanup Completed
Creating geohash_te
Creating hour_te
Creating roadtype_te
Creating day_te
OOF Target Encoding Completed
Feature Shape : (77299, 35)
String Conversion Completed
Label Encoding Completed


FOLD 1

Training CatBoost...
CatBoost R² : 0.9408342679967533

Training Tuned XGBoost...
XGBoost R² : 0.952048093494457

Training LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1349
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 35
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [In

In [ ]:
from scipy.stats import rankdata

In [ ]:
from scipy.stats import rankdata

# =========================================================
# RANKS
# =========================================================

cat_rank = rankdata(cat_preds)

xgb_rank = rankdata(xgb_preds)

lgb_rank = rankdata(lgb_preds)

# =========================================================
# WEIGHTED RANK BLEND
# =========================================================

rank_blend = (

    0.05 * cat_rank +

    0.70 * xgb_rank +

    0.25 * lgb_rank
)

# =========================================================
# ORIGINAL BEST ENSEMBLE
# =========================================================

original_blend = (

    0.05 * cat_preds +

    0.70 * xgb_preds +

    0.25 * lgb_preds
)

# =========================================================
# MATCH SCALE TO ORIGINAL BLEND
# =========================================================

rank_blend = (

    rank_blend - rank_blend.mean()

) / rank_blend.std()

final_rank_preds = (

    rank_blend * original_blend.std()

) + original_blend.mean()

print("Calibrated Rank Blend Created")

Calibrated Rank Blend Created


In [ ]:
# =========================================================
# CREATE SUBMISSION
# =========================================================

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': final_rank_preds
})

print(submission.head())

   Index    demand
0      0  0.103549
1      1 -0.023847
2      2 -0.035435
3      3 -0.046516
4      4  0.068245


In [ ]:
print(pd.Series(final_rank_preds).describe())

count    41778.000000
mean         0.127597
std          0.168240
min         -0.164402
25%         -0.019308
50%          0.127465
75%          0.272477
max          0.419594
dtype: float64


In [ ]:
# =========================================================
# SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_rank_ensemble.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\nSubmission Saved Successfully")

print(submission_path)